# Training ViT: From Scratch vs Fine-tuned

Training two Vision Transformers on a 10-class SUN397 subset to demonstrate the impact of pretraining. Compares a ViT trained from random initialization against a pretrained ViT fine-tuned via timm.

Reference: [Dosovitskiy et al., 2020](https://arxiv.org/abs/2010.11929) · [CS231N Lecture 12](https://cs231n.stanford.edu/slides/2024/lecture_12.pdf)

## Overview

**Builds:** `SceneSubset` · trained ViT from scratch · fine-tuned ViT via timm
**Concepts:** transfer learning · fine-tuning · data augmentation · cosine LR schedule · training loop
**Requires:** `02_vit_from_scratch.ipynb` — `VisionTransformer` architecture reused here
**References:** [Dosovitskiy et al., 2020](https://arxiv.org/abs/2010.11929) · [timm docs](https://huggingface.co/docs/timm)
**Output:** `models/vit_scratch_sun397.pth` · `models/vit_finetuned_sun397.pth` · `assets/training_curves.png`

---

**Contents**
1. [Part 1 — Dataset](#part-1--dataset)
2. [Part 2 — ViT from Scratch](#part-2--vit-from-scratch)
3. [Part 3 — Fine-tune Pretrained ViT](#part-3--fine-tune-pretrained-vit)
4. [Part 4 — Compare Results](#part-4--compare-results)
5. [Results Summary](#results-summary)

In [ ]:

# ── Colab setup ───────────────────────────────────────────────────────────────
# This cell is a no-op when running locally.
# On Colab: mounts Drive, installs packages, sets up paths so ../models etc. work.
import sys, os

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Install packages not in Colab base image
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'timm==1.0.25', 'einops==0.8.2'], check=True)

    # Create persistent directories on Drive
    DRIVE_BASE = '/content/drive/MyDrive/app-01'
    for d in ['models', 'assets', 'data']:
        os.makedirs(f'{DRIVE_BASE}/{d}', exist_ok=True)

    # Mirror repo structure locally and symlink to Drive for persistence
    os.makedirs('/content/app-01/notebooks', exist_ok=True)
    for d in ['models', 'assets', 'data']:
        dst = f'/content/app-01/{d}'
        if not os.path.exists(dst):
            os.symlink(f'{DRIVE_BASE}/{d}', dst)

    # Change to notebooks/ so all ../models, ../data, ../assets paths work unchanged
    os.chdir('/content/app-01/notebooks')
    print(f'Colab setup complete. CWD: {os.getcwd()}')
    print(f'Checkpoints and assets will persist to: {DRIVE_BASE}')
else:
    print('Local environment detected.')
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
import math
import time
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.datasets import CIFAR10
from torchvision import transforms
import timm
import matplotlib.pyplot as plt
import matplotlib
from PIL import Image

matplotlib.rcParams['figure.dpi'] = 120
torch.manual_seed(42)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

os.makedirs('../models', exist_ok=True)
os.makedirs('../assets', exist_ok=True)

## Part 1 — Dataset

CIFAR-10 contains 60,000 32×32 images across 10 classes (50k train / 10k test). Images are resized to 224×224 for ViT. Classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

Downloads reliably (~170 MB) and lets us focus on the training comparison without dataset setup issues.

In [ ]:
# Download CIFAR-10 — ~170 MB, skips if already present
print('Loading CIFAR-10...')
raw_train = CIFAR10(root='../data', train=True,  download=True)
raw_test  = CIFAR10(root='../data', train=False, download=True)
TARGET_CLASSES = raw_train.classes
print(f'Train: {len(raw_train):,} | Test: {len(raw_test):,}')
print(f'Classes: {TARGET_CLASSES}')

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_set = CIFAR10(root='../data', train=True,  download=False, transform=train_transform)
val_set   = CIFAR10(root='../data', train=False, download=False, transform=val_transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True,  num_workers=2, pin_memory=False)
val_loader   = DataLoader(val_set,   batch_size=64, shuffle=False, num_workers=2, pin_memory=False)

print(f'Train: {len(train_set):,} | Val: {len(val_set):,}')

In [ ]:
mean = np.array(IMAGENET_MEAN)
std  = np.array(IMAGENET_STD)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
shown = {i: False for i in range(10)}

for img, label in DataLoader(val_set, batch_size=1, shuffle=True):
    label = label.item()
    if not shown[label]:
        img_disp = img[0].permute(1, 2, 0).numpy() * std + mean
        img_disp = img_disp.clip(0, 1)
        ax = axes[label // 5][label % 5]
        ax.imshow(img_disp)
        ax.set_title(TARGET_CLASSES[label])
        ax.axis('off')
        shown[label] = True
    if all(shown.values()):
        break

plt.suptitle('One sample per class — CIFAR-10', y=1.02)
plt.tight_layout()
plt.savefig('../assets/sample_images.png', bbox_inches='tight', dpi=150)
plt.show()

---
[Back to top](#overview)

---
[Back to top](#overview)

## Part 2 — ViT from Scratch

Training a ViT from random initialization on our 10-class subset. We use a reduced config (6 layers, `d_model=256`, ~6M params) to make training tractable on a limited dataset without GPU hours.

**Why smaller?** ViT-B/16 (86M params) massively overfits on 7,500 images with random weights — the model memorizes training data before learning transferable features. A smaller ViT gives the architecture a fair chance and trains in minutes on MPS.

**Expected accuracy:** ~40–55% — limited by lack of pretraining, not architecture.

| Hyperparameter | Value |
|---|---|
| `patch_size` | 16 |
| `d_model` | 256 |
| `num_heads` | 8 |
| `num_layers` | 6 |
| Optimizer | AdamW, lr=1e-3, wd=0.01 |
| Scheduler | CosineAnnealingLR |
| Epochs | 15 |

In [ ]:
# VisionTransformer — copied from 02_vit_from_scratch.ipynb for self-contained training

def scaled_dot_product_attention(q, k, v, mask=None):
    d_k    = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    return torch.matmul(F.softmax(scores, dim=-1), v)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k, self.num_heads, self.d_model = d_model // num_heads, num_heads, d_model
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, x, mask=None):
        B, T, _ = x.shape
        q = self.split_heads(self.W_q(x))
        k = self.split_heads(self.W_k(x))
        v = self.split_heads(self.W_v(x))
        out = scaled_dot_product_attention(q, k, v, mask)
        return self.W_o(out.transpose(1, 2).reshape(B, T, self.d_model))


class PatchEmbedding(nn.Module):
    def __init__(self, image_size=224, patch_size=16, in_channels=3, d_model=256):
        super().__init__()
        self.num_patches = (image_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = MultiHeadAttention(d_model, num_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp   = nn.Sequential(
            nn.Linear(d_model, d_model * mlp_ratio), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * mlp_ratio, d_model), nn.Dropout(dropout),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.drop(self.attn(self.norm1(x)))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x


class VisionTransformer(nn.Module):
    def __init__(self, image_size=224, patch_size=16, in_channels=3, num_classes=10,
                 d_model=256, num_heads=8, num_layers=6, mlp_ratio=4, dropout=0.1):
        super().__init__()
        num_patches      = (image_size // patch_size) ** 2
        self.patch_embed = PatchEmbedding(image_size, patch_size, in_channels, d_model)
        self.cls_token   = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed   = nn.Parameter(torch.zeros(1, num_patches + 1, d_model))
        self.dropout     = nn.Dropout(dropout)
        self.blocks      = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        B   = x.shape[0]
        x   = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x   = self.dropout(torch.cat([cls, x], dim=1) + self.pos_embed)
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x)[:, 0])

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss   = criterion(logits, labels)
            total_loss += loss.item()
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / len(loader), correct / total


def run_training(model, train_loader, val_loader, optimizer, scheduler,
                 criterion, device, num_epochs, label, save_path):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0.0

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(va_acc)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), save_path)

        print(f'[{label}] Epoch {epoch:02d}/{num_epochs} | '
              f'train loss {tr_loss:.4f} acc {tr_acc:.3f} | '
              f'val loss {va_loss:.4f} acc {va_acc:.3f} | '
              f'{time.time() - t0:.1f}s')

    print(f'\nBest val acc: {best_val_acc:.3f} — saved to {save_path}')
    return history

In [ ]:
scratch_vit = VisionTransformer(
    image_size=224, patch_size=16, in_channels=3,
    num_classes=10, d_model=256, num_heads=8, num_layers=6
).to(device)

n_params = sum(p.numel() for p in scratch_vit.parameters())
print(f'Scratch ViT parameters: {n_params:,}')

criterion    = nn.CrossEntropyLoss()
optimizer_sc = torch.optim.AdamW(scratch_vit.parameters(), lr=1e-3, weight_decay=0.01)
scheduler_sc = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_sc, T_max=15)

history_scratch = run_training(
    scratch_vit, train_loader, val_loader,
    optimizer_sc, scheduler_sc, criterion, device,
    num_epochs=15, label='ViT-Scratch',
    save_path='../models/vit_scratch_sun397.pth'
)

---
[Back to top](#overview)

## Part 3 — Fine-tune Pretrained ViT

Loading `vit_base_patch16_224` from timm — pretrained on ImageNet-21k (~14M images) then fine-tuned on ImageNet-1k. We replace the classification head with a 10-class linear layer and fine-tune all weights at a low learning rate.

**Why fine-tune all layers?** Scene classification requires spatial understanding across the whole image. Freezing the backbone and only training the head works for object tasks but underperforms for scenes where global context across all patches matters. Full fine-tuning at a small lr retains pretrained representations while adapting them to our classes.

**Expected accuracy:** ~75–85% — 14M pretraining images provide rich scene representations that 7,500 images alone cannot build.

| Hyperparameter | Value |
|---|---|
| Model | `vit_base_patch16_224` (timm, pretrained) |
| Params | ~86M |
| Optimizer | AdamW, lr=1e-4, wd=0.01 |
| Scheduler | CosineAnnealingLR |
| Epochs | 5 |

In [ ]:
finetuned_vit = timm.create_model(
    'vit_base_patch16_224',
    pretrained=True,
    num_classes=10
).to(device)

n_params = sum(p.numel() for p in finetuned_vit.parameters())
print(f'Fine-tuned ViT parameters: {n_params:,}')

optimizer_ft = torch.optim.AdamW(finetuned_vit.parameters(), lr=1e-4, weight_decay=0.01)
scheduler_ft = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=5)

history_ft = run_training(
    finetuned_vit, train_loader, val_loader,
    optimizer_ft, scheduler_ft, criterion, device,
    num_epochs=5, label='ViT-Finetuned',
    save_path='../models/vit_finetuned_sun397.pth'
)

---
[Back to top](#overview)

## Part 4 — Compare Results

Side-by-side learning curves for both models. The accuracy gap visualizes the transfer learning advantage — pretraining on 14M images gives the fine-tuned model a head start that 15 epochs of scratch training on 7,500 images cannot match.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_sc = range(1, len(history_scratch['val_acc']) + 1)
epochs_ft = range(1, len(history_ft['val_acc']) + 1)

ax = axes[0]
ax.plot(epochs_sc, history_scratch['train_acc'], '--', color='steelblue',  label='Scratch — train')
ax.plot(epochs_sc, history_scratch['val_acc'],   '-',  color='steelblue',  label='Scratch — val')
ax.plot(epochs_ft, history_ft['train_acc'],      '--', color='darkorange', label='Fine-tuned — train')
ax.plot(epochs_ft, history_ft['val_acc'],        '-',  color='darkorange', label='Fine-tuned — val')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(epochs_sc, history_scratch['train_loss'], '--', color='steelblue',  label='Scratch — train')
ax.plot(epochs_sc, history_scratch['val_loss'],   '-',  color='steelblue',  label='Scratch — val')
ax.plot(epochs_ft, history_ft['train_loss'],      '--', color='darkorange', label='Fine-tuned — train')
ax.plot(epochs_ft, history_ft['val_loss'],        '-',  color='darkorange', label='Fine-tuned — val')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('ViT from Scratch vs Fine-tuned — CIFAR-10', y=1.02)
plt.tight_layout()
plt.savefig('../assets/training_curves.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
scratch_best   = max(history_scratch['val_acc'])
finetuned_best = max(history_ft['val_acc'])

print('Results Summary')
print('=' * 52)
print(f'{"Model":<22} {"Params":>8} {"Best Val Acc":>12}')
print('-' * 52)
print(f'{"ViT from scratch":<22} {"~6M":>8} {scratch_best:>11.1%}')
print(f'{"ViT fine-tuned (timm)":<22} {"~86M":>8} {finetuned_best:>11.1%}')
print('=' * 52)
print(f'\nTransfer learning gap: +{finetuned_best - scratch_best:.1%}')
print(f'\nFine-tuned checkpoint: ../models/vit_finetuned_sun397.pth')
print('Load this in 04_gradcam_visualization.ipynb')

---
[Back to top](#overview)

## Results Summary

*Fill in after running.*

| Model | Params | Best Val Accuracy |
|---|---|---|
| ViT from scratch | ~6M | — |
| ViT fine-tuned (timm) | ~86M | — |

The accuracy gap demonstrates why pretraining matters: ImageNet weights encode rich visual representations that 7,500 scene images cannot build from random initialization in 15 epochs.

The fine-tuned checkpoint (`models/vit_finetuned_sun397.pth`) is used in notebooks 04 and 05.

---
**Next:** [`04_gradcam_visualization.ipynb`](04_gradcam_visualization.ipynb) — apply Grad-CAM and attention rollout to the fine-tuned model to visualize which image regions drive scene predictions